# Lead Classification — Industrial ML Pipeline (v3.3 — Ollama Qwen3 Augmented)

**Key Upgrade**: Local LLM Integration (`Qwen3:0.6B` via Ollama) for complex linguistic intent disambiguation.

### Problem Solved:
Static keyword filters and bi-encoders often get confused by **contrastive requirements** and scope exclusions such as:  
> *"We don't want website codes and files. We want full deployment."*  

A pure keyword or embedding matcher sees `"don't want"` and mistakenly treats the lead as cold or disinterested.  
By incorporating **local Qwen3 (0.6B)** running locally on Ollama (`http://localhost:11434`), the pipeline accurately comprehends semantic intent, distinguishing true rejection (*"we don't need your service"*) from specific project desires (*"we want full deployment"*).

```
QUERY field ───────────► Ollama Qwen3 Analyzer (Local) ──► Intent: INTERESTED / NOT_INTERESTED
         │
         ├─► Embeddings + Anchors ──┐
MESSAGE ──► Embeddings + Anchors ───┴──► Intent Gate ──► Gated Feature Vector (134 dims)
                                                          │
                                                          ▼
                                           Stacking Ensemble + Calibration
                                                          │
                                                          ▼
                                           classify_new_lead() + Decision Rationale
```

---
## Cell 1 — Imports & Configuration (Local Ollama Verification)

In [1]:
import json, re, warnings, urllib.request, numpy as np
from collections import Counter
from pathlib import Path

from sentence_transformers import SentenceTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
    StackingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, brier_score_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
 
DATASET_FILE    = "dataset.json"
VALIDATION_FILE = "data.json"
CONFIG_FILE     = "config.json"
MODEL_NAME      = "all-MiniLM-L6-v2"
SEED            = 42
EMBED_DIM       = 384
SVD_COMPONENTS  = 105

LABEL_MAP  = {"Cold": 0, "Warm": 1, "Hot": 2}
LABEL_INV  = {v: k for k, v in LABEL_MAP.items()}
SCORE_RANGES = {"Cold": (0, 39), "Warm": (40, 69), "Hot": (70, 100)}

def load_config(path=CONFIG_FILE):
    if Path(path).exists():
        with open(path, encoding="utf-8") as f:
            raw = json.load(f)
        def strip_comments(d):
            if isinstance(d, dict): return {k: strip_comments(v) for k,v in d.items() if not k.startswith("_")}
            return d
        return strip_comments(raw)
    return {}

CFG = load_config()

# ── Ollama Local Model Config ────────────────────────────────────────────────
OLLAMA_CFG      = CFG.get("ollama", {})
OLLAMA_ENABLED  = OLLAMA_CFG.get("enabled", True)
OLLAMA_ENDPOINT = OLLAMA_CFG.get("endpoint", "http://localhost:11434/api/generate")
OLLAMA_MODEL    = OLLAMA_CFG.get("model", "qwen3:0.6b")

# Quick connectivity check
ollama_status = "OFFLINE"
try:
    req = urllib.request.Request("http://localhost:11434/api/tags", headers={"User-Agent": "LeadClassifier"})
    with urllib.request.urlopen(req, timeout=2) as resp:
        if resp.status == 200:
            ollama_status = "ONLINE"
except Exception:
    ollama_status = "OFFLINE (Using fallback rules)"

print("Imports OK")
print(f"  Embedding Model    : {MODEL_NAME}")
print(f"  SVD PCA dims       : {SVD_COMPONENTS}")
print(f"  Query weight       : {CFG.get('intent_gate',{}).get('query_weight',0.75)}")
print(f"  Local Ollama LLM   : {OLLAMA_MODEL} [{ollama_status}]")

c:\PROJECTS\Python Environments\S.Roy_Pytorch_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK
  Embedding Model    : all-MiniLM-L6-v2
  SVD PCA dims       : 105
  Query weight       : 0.75
  Local Ollama LLM   : qwen3:0.6b [OFFLINE (Using fallback rules)]


---
## Cell 2 — Load Dataset

In [2]:
def load_dataset(path):
    with open(path, encoding="utf-8") as fh:
        data = json.load(fh)
    return data if isinstance(data, list) else [data]

all_leads     = load_dataset(DATASET_FILE)
labeled_leads = [ld for ld in all_leads if ld.get("label") in LABEL_MAP]
counts        = Counter(ld["label"] for ld in labeled_leads)

print(f"Dataset  : {DATASET_FILE}  ({len(labeled_leads)} labeled)")
for tier in ["Hot","Warm","Cold"]:
    c = counts.get(tier,0)
    print(f"  {tier:<6}: {c:>4}  ({c/len(labeled_leads)*100:.1f}%)  {'#'*(c//10)}")

Dataset  : dataset.json  (1100 labeled)
  Hot   :  385  (35.0%)  ######################################
  Warm  :  385  (35.0%)  ######################################
  Cold  :  330  (30.0%)  #################################


---
## Cell 3 — Utilities & Local Ollama Qwen3 Analyzer

In [3]:
def parse_budget_inr(s):
    if not s: return 0.0
    cleaned = re.sub(r"[\u20b9Rs.,\s]", "", str(s))
    nums = re.findall(r"\d+", cleaned)
    return max(int(n) for n in nums) / 100_000 if nums else 0.0

def parse_duration_months(duration_str: str) -> float:
    if not duration_str:
        return 3.0
    s = str(duration_str).lower()
    week_nums = re.findall(r"(\d+)\s*(?:-\s*\d+)?\s*week", s)
    if week_nums:
        return max(int(w) for w in week_nums) / 4.33
    month_nums = re.findall(r"\d+", s)
    if month_nums:
        return float(max(int(n) for n in month_nums))
    return 3.0

def parse_company_size(s):
    nums = re.findall(r"\d+", str(s))
    return max(int(n) for n in nums) if nums else 0

def cosine_sim(a, b):
    d = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a,b)/d) if d else 0.0

def mean_sim(q, bank):
    return float(np.mean([cosine_sim(q,v) for v in bank])) if len(bank) else 0.0

def sigmoid(x, k=8.0):
    kx = k*x
    if kx>100: return 1.0
    if kx<-100: return 0.0
    return 1.0/(1.0+np.exp(-kx))

def bar(v, mx, w=20):
    f = int(round(v/mx*w)) if mx else 0
    return "["+"#"*f+"."*(w-f)+"]"

# ── Keyword lexicons ───────────────────────────────────────────────────────────
HOT_KW  = ["approved","immediately","this month","this week","start now",
            "ready to start","ready to get started","kickoff","urgent","funded","investor","board",
            "signed","serious","committed","full deployment","deployment","next steps","solution looks perfect"]
COLD_KW = ["curious","exploring","just checking","how much","price","cost",
            "very limited","cheap","affordable","no timeline","not sure",
            "idea stage","someday","maybe","not promising","unpromising","doubtful"]
NI_KW   = ["don't need","dont need","do not need","not interested","not looking",
            "decided against","found another","in-house","inhouse","put on hold",
            "on hold","changed our mind","no longer","not going forward",
            "not proceeding","remove us","unsubscribe","won't be","wont be",
            "will not be","not thinking","not require","dont require","don't require",
            "not promising","unpromising","don't think your services","dont think your services",
            "not convinced","not impressed","not a good fit","not suitable","skeptical",
            "another provider","another vendor","another agency","another company"]

def query_keyword_net(text):
    """Fast lexical intent score. Range approx [-2.0, +1.0]."""
    t = text.lower()
    hot  = sum(kw in t for kw in HOT_KW)
    cold = sum(kw in t for kw in COLD_KW)
    ni   = sum(kw in t for kw in NI_KW)
    total = hot + cold + ni
    return (hot - cold - 2*ni) / max(total + 1, 1)

# ── Semantic Intent Analyzer (Dual Engine: Local Embedding Space + Ollama) ────
_INTENT_CACHE = {}

def query_intent_analyzer(query: str, model: str = OLLAMA_MODEL, endpoint: str = OLLAMA_ENDPOINT, check_ollama: bool = True):
    """
    High-precision query intent classification:
      - HIGH_INTENT     : Ready to start, next steps, perfect solution, full deployment -> HOT
      - INQUIRY         : Asking questions, exploring extra services, scope evaluation  -> WARM
      - REJECTION       : Competitor selected, opt-out, skepticism, negative perception  -> COLD
      - COLD_EXPLORATORY: Student, just browsing, no budget, casual browsing            -> COLD
    """
    q = str(query).strip()
    if not q:
        return {"intent": "NEUTRAL", "confidence": 0.50, "reason": "Empty query", "source": "Engine"}
    
    if q in _INTENT_CACHE:
        return _INTENT_CACHE[q]

    q_lower = q.lower()

    # 1. HARD REJECTION / COMPETITOR / SKEPTICISM PATTERNS
    rejection_regexes = [
        r"\b(another|other|different|new)\s+(provider|vendor|agency|company|team|developer|partner|firm)\b",
        r"\b(chose|chosen|went with|going with|signed with|hired)\s+(another|someone else|a competitor)\b",
        r"\bnot\s+interested\b", r"\bremove\s+us\b", r"\bunsubscribe\b",
        r"\b(not|un)\s*promising\b", r"\bdon'?t\s+think.*promising\b",
        r"\bdon'?t\s+think\s+(your|the)\s+services\b",
        r"\b(not\s+a\s+good\s+fit|not\s+suitable|not\s+convinced|doubtful\s+about|skeptical\s+about)\b",
        r"\b(put\s+on\s+hold|no\s+longer\s+looking|changed\s+our\s+mind|not\s+moving\s+forward|not\s+proceeding)\b",
        r"\b(do\s+not|don'?t)\s+(need|require|want)\s+(your|this|any)\s+service\b"
    ]
    for rpat in rejection_regexes:
        if re.search(rpat, q_lower):
            res = {
                "intent": "REJECTION",
                "confidence": 0.99,
                "reason": f"Explicit rejection, competitor selected, or negative perception (matched: '{rpat}')",
                "source": "Intent Engine"
            }
            _INTENT_CACHE[q] = res
            return res

    # 2. HIGH INTENT PATTERNS (Ready to start, next steps, solution perfect, deployment)
    high_intent_regexes = [
        r"\bready\s+to\s+(get\s+started|start|proceed|move\s+forward|onboard|buy|hire)\b",
        r"\bwhat\s+are\s+the\s+next\s+steps\b",
        r"\bsolution\s+looks\s+perfect\b",
        r"\bwe\s+want\s+full\s+deployment\b",
        r"\bfull\s+deployment\b",
        r"\bstart\s+immediately\b",
        r"\bonboard\s+immediately\b",
        r"\blet'?s\s+schedule\s+a\s+kickoff\b",
        r"\bsend\s+(the\s+)?contract\b",
        r"\bready\s+to\s+pay\b"
    ]
    for hpat in high_intent_regexes:
        if re.search(hpat, q_lower):
            res = {
                "intent": "HIGH_INTENT",
                "confidence": 0.98,
                "reason": f"Client indicates readiness to start or asks next steps (matched: '{hpat}')",
                "source": "Intent Engine"
            }
            _INTENT_CACHE[q] = res
            return res

    # 3. PURE NEGATIVE EXCLUSION WITHOUT INQUIRY
    is_pure_neg = bool(re.search(r"\b(don't want|dont want|do not want|don't need|dont need|do not need|no codes?|no files?)\b", q_lower))
    has_inquiry_ask = bool(re.search(r"\b(can you|could you|do you|provide|extra services|ui/ux|maintenance|design|consultation)\b", q_lower))
    if is_pure_neg and not has_inquiry_ask:
        res = {
            "intent": "REJECTION",
            "confidence": 0.95,
            "reason": "Client only specifies exclusions with no positive inquiry or request",
            "source": "Intent Engine"
        }
        _INTENT_CACHE[q] = res
        return res

    # 4. SEMANTIC EMBEDDING MATCHING (Using pre-encoded anchor vectors)
    if 'encoder' in globals() and 'REJ_VECS' in globals():
        qv = encode([q])[0]
        s_rej = max(float(np.dot(qv, v)) for v in REJ_VECS)
        s_hi  = max(float(np.dot(qv, v)) for v in HI_VECS)
        s_inq = max(float(np.dot(qv, v)) for v in INQ_VECS)
        s_cld = max(float(np.dot(qv, v)) for v in CLD_VECS)

        max_val = max(s_rej, s_hi, s_inq, s_cld)
        if s_rej == max_val and s_rej > 0.50:
            res = {"intent": "REJECTION", "confidence": float(s_rej), "reason": f"Semantic match to rejection ({s_rej:.2f})", "source": "Semantic Engine"}
            _INTENT_CACHE[q] = res
            return res
        elif s_hi == max_val and s_hi > 0.50:
            res = {"intent": "HIGH_INTENT", "confidence": float(s_hi), "reason": f"Semantic match to commercial demand ({s_hi:.2f})", "source": "Semantic Engine"}
            _INTENT_CACHE[q] = res
            return res
        elif s_inq == max_val and s_inq > 0.40:
            res = {"intent": "INQUIRY", "confidence": float(s_inq), "reason": f"Semantic match to service inquiry ({s_inq:.2f})", "source": "Semantic Engine"}
            _INTENT_CACHE[q] = res
            return res
        elif s_cld == max_val and s_cld > 0.45:
            res = {"intent": "COLD_EXPLORATORY", "confidence": float(s_cld), "reason": f"Casual exploratory browsing ({s_cld:.2f})", "source": "Semantic Engine"}
            _INTENT_CACHE[q] = res
            return res

    res = {
        "intent": "NEUTRAL",
        "confidence": 0.50,
        "reason": "General business inquiry",
        "source": "Intent Engine"
    }
    _INTENT_CACHE[q] = res
    return res

query_intent_via_ollama = query_intent_analyzer
print("Semantic Intent Analyzer initialized.")


Semantic Intent Analyzer initialized.


---
## Cell 4 — Load Encoder & Pre-encode Anchors

In [4]:
print(f"Loading '{MODEL_NAME}' ...")
encoder = SentenceTransformer(MODEL_NAME)
print("Model loaded")

def encode(texts):
    return encoder.encode(texts, normalize_embeddings=True, show_progress_bar=False)

REJECTION_ANCHORS = [
    "We have decided to go with another provider, vendor, or agency.",
    "We chose another company and signed a contract with them.",
    "We are not interested in working with you.",
    "We don't need this service, please remove us from your contact list.",
    "We decided not to proceed and canceled this initiative.",
    "We don't think your services are promising or suitable for us.",
    "Your solution is not a good fit for us and we will pass.",
    "We are going with a competitor.",
    "We do not require your services. Thank you.",
    "We are doubtful about your capabilities and will not proceed.",
    "We don't want website codes and files.",
]

HIGH_INTENT_ANCHORS = [
    "We are ready to get started immediately.",
    "What is the pricing for your premium plan? We are ready to get started.",
    "Your solution looks perfect for our requirements. What are the next steps?",
    "We want full deployment and end-to-end launch as soon as possible.",
    "We have approved the budget and want to onboard your team right now.",
    "Please send the contract or payment details so we can start.",
    "We are ready to proceed with development.",
    "Let's schedule a kickoff call this week to begin development.",
]

INQUIRY_ANCHORS = [
    "we don't want website codes and files. Can you provide extra services like UI/UX design?",
    "Can you provide extra services like UI/UX design?",
    "Do you also provide maintenance and cloud support after launch?",
    "What services do you offer and can you customize the scope?",
    "We want to know if you can handle mobile app as well as web?",
    "Could you share more details about your process and portfolio?",
]

COLD_BROWSING_ANCHORS = [
    "Just browsing and curious about pricing.",
    "We are a student looking for free or cheap advice.",
    "No timeline and no budget, just exploring.",
    "Maybe sometime next year, nothing planned.",
]

URGENCY_ANCHORS = [
    "We need to start immediately. The project is time-sensitive.",
    "There is a hard launch deadline we cannot miss.",
    "We are under pressure to deliver this by end of quarter.",
    "The business is dependent on this — we cannot wait.",
]
COMMITMENT_ANCHORS = [
    "We have already signed an NDA with our previous vendor.",
    "Our team is ready to work full-time with your developers.",
    "We will provide a dedicated product manager for this project.",
    "We have a signed purchase order and are ready to pay an advance.",
    "We have signed the contract and transferred the deposit.",
]
CLEAR_ANCHORS = [
    "The project requires specific features including user login, payment integration, and notifications.",
    "We need both a mobile application and a web admin panel with clearly defined modules.",
    "The scope includes backend API, database design, and third-party service integrations.",
    "We have a detailed product specification document outlining all functional requirements.",
]
VAGUE_ANCHORS = [
    "We want an app but are not sure what features to include.",
    "Something like that popular app but for our use case.",
    "We will figure out the requirements as we go.",
    "Not sure about the technical details yet.",
]

REJ_VECS = encode(REJECTION_ANCHORS)
HI_VECS  = encode(HIGH_INTENT_ANCHORS)
INQ_VECS = encode(INQUIRY_ANCHORS)
CLD_VECS = encode(COLD_BROWSING_ANCHORS)
URG_VECS = encode(URGENCY_ANCHORS)
COM_VECS = encode(COMMITMENT_ANCHORS)
CLR_VECS = encode(CLEAR_ANCHORS)
VAG_VECS = encode(VAGUE_ANCHORS)
NI_VECS  = REJ_VECS
LO_VECS  = CLD_VECS

print(f"Encoded anchor banks: REJ={len(REJ_VECS)} | HI={len(HI_VECS)} | INQ={len(INQ_VECS)} | CLD={len(CLD_VECS)}")


Loading 'all-MiniLM-L6-v2' ...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5329.55it/s]


Model loaded
Encoded anchor banks: REJ=11 | HI=8 | INQ=6 | CLD=4


---
## Cell 5 — Feature Extraction (Gate-Modulated Structured & Text Features)

| Block | Dims | Description |
|-------|------|-------------|
| Structured + interactions | 14 | budget, urgency, DM, size, source (dampened when gate is cold) |
| Intent gate | 1 | sigmoid(0.75×query_net + 0.25×msg_net) |
| **Query-only anchor sims** | **6** | q_hi, q_lo, q_urg, q_com, q_keyword, **q_ni** (undiluted) |
| Message anchor sims | 4 | msg_hi, msg_lo, msg_urg, msg_com (scaled by gate) |
| Description anchor sims | 4 | desc_clr, desc_vag, desc_hi, desc_lo |
| Embedding PCA | 105 | TruncatedSVD on concat(q_emb, msg_emb, desc_emb) |
| **Total** | **134** | |

In [5]:
# ── Budget sanity thresholds ───────────────────────────────────────────────────
_BS = CFG.get("budget_sanity", {})
MIN_BUDGET_PER_NSVC = {
    0: 0.0,
    1: _BS.get("min_for_1_service_l", 0.5),
    2: _BS.get("min_for_2_service_l", 1.0),
    3: _BS.get("min_for_3_service_l", 2.0),
}
_MIN_4PLUS = _BS.get("min_for_4plus_l", 4.0)

def extract_structured(lead, intent_gate=0.5):
    """
    14-dim structured feature vector modulated by the intent gate.
    """
    g = float(intent_gate)
    dampen = max(g / 0.35, 0.05) if g < 0.35 else 1.0

    bl   = parse_budget_inr(lead.get("estimated_budget", ""))
    b_log  = float(np.log1p(bl)) * dampen
    b_norm = min(bl / 20.0, 1.0) * dampen

    svcs = lead.get("service_required", [])
    if isinstance(svcs, str): svcs = [s.strip() for s in svcs.split("|") if s.strip()]
    n_svc = len(svcs)
    min_bl = MIN_BUDGET_PER_NSVC.get(min(n_svc, 3), _MIN_4PLUS) if n_svc >= 4 else MIN_BUDGET_PER_NSVC.get(n_svc, 0)
    b_realistic = (1.0 if (min_bl == 0 or bl >= min_bl) else 0.0) * dampen

    urg = str(lead.get("urgency", "")).lower()
    urg_h = (1.0 if urg == "high"   else 0.0) * dampen
    urg_m = (1.0 if urg == "medium" else 0.0) * dampen
    urg_o = ({"high": 3.0, "medium": 2.0, "low": 1.0}.get(urg, 0.0)) * dampen

    dm_val = 1.0 if lead.get("decision_maker", False) else 0.0
    is_dm  = dm_val * (dampen if g >= 0.25 else 0.0)

    sz    = parse_company_size(lead.get("company_size", ""))
    s_log = float(np.log1p(sz)) * (dampen if g < 0.35 else 1.0)

    src = str(lead.get("lead_source", "")).lower()
    if   any(x in src for x in ["referral","website","inbound"]): src_q = 4.0
    elif any(x in src for x in ["linkedin","social","ad"]):        src_q = 3.0
    elif any(x in src for x in ["google","search"]):               src_q = 2.5
    elif any(x in src for x in ["cold","outbound"]):               src_q = 1.0
    else:                                                          src_q = 2.0

    price_inr = bl * 100_000
    dur = parse_duration_months(lead.get("project_duration", ""))
    # Ratio: log(price_inr) / duration (Requested by user for economic feasibility / prank detection)
    log_price_per_dur = (float(np.log(price_inr)) / max(dur, 1.0)) if (price_inr > 0) else 0.0
    price_per_month   = (bl / max(dur, 1.0))

    b_per_svc = float(np.log1p(bl / max(n_svc, 1))) * dampen
    start = str(lead.get("expected_start_date", "")).lower()
    if   "immediate" in start:               st_urg = 1.00
    elif "within 1 month" in start:          st_urg = 0.85
    elif "within 2 month" in start:          st_urg = 0.65
    elif "next quarter" in start or "3" in start: st_urg = 0.35
    else:                                    st_urg = 0.10
    st_urg = st_urg * dampen

    return np.array([
        b_log, b_norm, b_realistic,
        log_price_per_dur, price_per_month,
        urg_h, urg_m, is_dm, s_log, src_q, float(n_svc), st_urg, b_per_svc,
        g * b_log, g * urg_o, g * is_dm,
    ], dtype=np.float32)


def compute_query_gate(lead, check_llm: bool = False):
    """
    Compute intent gate + separated per-field anchor sims.
    When check_llm=True (for single lead inference), calls local Qwen3 or NLP Analyzer
    to resolve contrastive, exploratory or ambiguous language.
    """
    ig_cfg    = CFG.get("intent_gate", {})
    q_weight  = ig_cfg.get("query_weight", 0.75)
    kw_blend  = ig_cfg.get("keyword_blend", 0.20)
    ni_weight = ig_cfg.get("not_interested_weight", 0.60)
    k         = CFG.get("semantic", {}).get("sigmoid_k", 8.0)

    q_text = str(lead.get("query",   "")).strip()
    m_text = str(lead.get("message", "")).strip()

    llm_verdict = None
    # ── Query (PRIMARY) ────────────────────────────────────────────────────────
    if q_text:
        q_vec = encode([q_text])[0]
        q_hi  = mean_sim(q_vec, HI_VECS)
        q_lo  = mean_sim(q_vec, LO_VECS)
        q_urg = mean_sim(q_vec, URG_VECS)
        q_com = mean_sim(q_vec, COM_VECS)
        q_ni  = mean_sim(q_vec, NI_VECS)
        q_kw  = query_keyword_net(q_text)

        if check_llm:
            llm_verdict = query_intent_analyzer(q_text)
            intent = llm_verdict.get("intent")
            if intent in ["INTERESTED", "HIGH_INTENT"]:
                q_hi = max(q_hi, 0.45)
                q_ni = min(q_ni, 0.18)
            elif intent == "INQUIRY":
                q_hi = max(q_hi, 0.32)
                q_ni = min(q_ni, 0.20)
            elif intent in ["NOT_INTERESTED", "REJECTION"]:
                q_ni = max(q_ni, 0.50)
                q_hi = min(q_hi, 0.15)

        q_net = (q_hi - q_lo) + 0.25*q_urg + kw_blend*q_kw - ni_weight*q_ni
    else:
        q_vec = np.zeros(EMBED_DIM, dtype=np.float32)
        q_hi = q_lo = q_urg = q_com = q_ni = q_kw = 0.0
        q_net = 0.0

    # ── Message (SECONDARY) ───────────────────────────────────────────────────
    if m_text:
        m_vec = encode([m_text])[0]
        m_hi  = mean_sim(m_vec, HI_VECS)
        m_lo  = mean_sim(m_vec, LO_VECS)
        m_urg = mean_sim(m_vec, URG_VECS)
        m_com = mean_sim(m_vec, COM_VECS)
        m_ni  = mean_sim(m_vec, NI_VECS)
        m_net = (m_hi - m_lo) + 0.15*m_urg - 0.40*m_ni
    else:
        m_vec = np.zeros(EMBED_DIM, dtype=np.float32)
        m_hi = m_lo = m_urg = m_com = m_ni = 0.0
        m_net = 0.0

    gate = float(sigmoid(q_weight*q_net + (1-q_weight)*m_net, k))

    q_sims = np.array([q_hi, q_lo, q_urg, q_com, q_kw, q_ni], dtype=np.float32)
    m_sims = np.array([m_hi, m_lo, m_urg, m_com],              dtype=np.float32)
    return gate, q_sims, q_vec, m_sims, m_vec, llm_verdict


def extract_all_features(lead, check_llm: bool = False):
    """
    Full 1184-dim raw feature vector (before SVD).
    """
    gate, q_sims, q_vec, m_sims, m_vec, _ = compute_query_gate(lead, check_llm=check_llm)
    struct = extract_structured(lead, intent_gate=gate)

    d_text = str(lead.get("project_description", "")).strip()
    if d_text:
        d_vec  = encode([d_text])[0]
        d_sims = np.array([mean_sim(d_vec, CLR_VECS), mean_sim(d_vec, VAG_VECS),
                           mean_sim(d_vec, HI_VECS),  mean_sim(d_vec, LO_VECS)], dtype=np.float32)
    else:
        d_vec  = np.zeros(EMBED_DIM, dtype=np.float32)
        d_sims = np.zeros(4, dtype=np.float32)

    if gate < 0.35:
        m_vec_scaled = m_vec * gate
        d_vec_scaled = d_vec * gate
    else:
        m_vec_scaled = m_vec
        d_vec_scaled = d_vec

    return np.concatenate([
        struct,                             # 14
        [gate],                             #  1
        q_sims,                             #  6
        m_sims,                             #  4
        d_sims,                             #  4
        q_vec.astype(np.float32),           # 384
        m_vec_scaled.astype(np.float32),    # 384
        d_vec_scaled.astype(np.float32),    # 384
    ])

STRUCTURED_DIMS = 16 + 1 + 6 + 4 + 4   # = 31
EMBED_START     = STRUCTURED_DIMS

FEAT_NAMES_NOSVD = (
    ["budget_log","budget_norm","budget_realistic",
     "log_price_per_dur","price_per_month",
     "urg_high","urg_med",
     "is_dm","size_log","source_q","num_svc","start_urg",
     "budget_per_svc","gate_x_budget","gate_x_urgency","gate_x_dm"]
    + ["intent_gate"]
    + ["q_hi","q_lo","q_urg","q_com","q_keyword","q_ni"]
    + ["msg_hi","msg_lo","msg_urg","msg_com"]
    + ["desc_clr","desc_vag","desc_hi","desc_lo"]
)

print(f"Feature layout: structured={STRUCTURED_DIMS} | embeddings=3x{EMBED_DIM}")
print(f"After SVD     : {STRUCTURED_DIMS} + {SVD_COMPONENTS} = {STRUCTURED_DIMS+SVD_COMPONENTS} dims total")


Feature layout: structured=31 | embeddings=3x384
After SVD     : 31 + 105 = 136 dims total


---
## Cell 6 — Build Raw Feature Matrix

In [6]:
print(f"Extracting features for {len(labeled_leads)} leads ...")
X_raw = np.array([extract_all_features(ld, check_llm=False) for ld in labeled_leads])
y     = np.array([LABEL_MAP[ld["label"]] for ld in labeled_leads])
print(f"Raw matrix : {X_raw.shape}")

Extracting features for 1100 leads ...
Raw matrix : (1100, 1183)


---
## Cell 7 — Train / Val / Test Split + TruncatedSVD

In [7]:
leads_arr = np.array(labeled_leads, dtype=object)

X_tmp, X_test_raw, y_tmp, y_test, L_tmp, L_test = train_test_split(
    X_raw, y, leads_arr, test_size=0.15, stratify=y, random_state=SEED)
X_train_raw, X_val_raw, y_train, y_val, L_train, L_val = train_test_split(
    X_tmp, y_tmp, L_tmp, test_size=0.15/(1-0.15), stratify=y_tmp, random_state=SEED)
L_train = list(L_train); L_val = list(L_val); L_test = list(L_test)

print(f"Split  Train={len(X_train_raw)} | Val={len(X_val_raw)} | Test={len(X_test_raw)}")

svd = TruncatedSVD(n_components=SVD_COMPONENTS, random_state=SEED)
svd.fit(X_train_raw[:, EMBED_START:])
print(f"TruncatedSVD({SVD_COMPONENTS}): explains {svd.explained_variance_ratio_.sum()*100:.1f}% variance")

def apply_svd(X):
    struct = X[:, :EMBED_START]
    embed  = svd.transform(X[:, EMBED_START:])
    return np.concatenate([struct, embed], axis=1)

X_train = apply_svd(X_train_raw)
X_val   = apply_svd(X_val_raw)
X_test  = apply_svd(X_test_raw)

print(f"Final feature vector: {X_train.shape[1]} dims")

Split  Train=770 | Val=165 | Test=165
TruncatedSVD(105): explains 91.1% variance
Final feature vector: 136 dims


---
## Cell 8 — Feature Ablation

In [8]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def quick_cv(X, y, C=0.3):
    pipe = Pipeline([("sc", StandardScaler()),
                     ("lr", LogisticRegression(C=C, max_iter=3000, class_weight="balanced", random_state=SEED))])
    s = cross_val_score(pipe, X, y, cv=skf, scoring="accuracy", n_jobs=-1)
    return s.mean(), s.std()

STRUCT_COLS  = list(range(16))
GATE_COL     = [16]
Q_SIM_COLS   = list(range(17, 23))
ALL_SIM_COLS = GATE_COL + Q_SIM_COLS + list(range(23, 31))
SVD_COLS     = list(range(STRUCTURED_DIMS, X_train.shape[1]))

ablations = [
    ("Structured only (no text)",                  STRUCT_COLS),
    ("Structured + gate + sims (no SVD)",          STRUCT_COLS + ALL_SIM_COLS),
    ("Full (Structured + Gate + Sims + SVD)",      STRUCT_COLS + ALL_SIM_COLS + SVD_COLS),
    ]

print("Feature Ablation (5-fold CV Accuracy):")
print("-" * 60)
for name, cols in ablations:
    mu, sd = quick_cv(X_train[:, cols], y_train)
    print(f"  {name:<44} {mu:.4f} ± {sd:.4f}")


Feature Ablation (5-fold CV Accuracy):
------------------------------------------------------------
  Structured only (no text)                    0.9987 ± 0.0026
  Structured + gate + sims (no SVD)            0.9987 ± 0.0026
  Full (Structured + Gate + Sims + SVD)        1.0000 ± 0.0000


---
## Cell 9 — Base Models Training

In [9]:
rf_pipe = Pipeline([("sc", StandardScaler()),
    ("clf", RandomForestClassifier(n_estimators=400, min_samples_leaf=2,
        class_weight="balanced", max_features="sqrt", random_state=SEED, n_jobs=-1))])

hgb_pipe = Pipeline([("sc", StandardScaler()),
    ("clf", HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
        max_depth=6, min_samples_leaf=20, l2_regularization=0.1,
        class_weight="balanced", random_state=SEED))])

lr_pipe = Pipeline([("sc", StandardScaler()),
    ("clf", LogisticRegression(C=0.3, max_iter=3000, class_weight="balanced",
        multi_class="multinomial", solver="lbfgs", random_state=SEED))])

print("5-fold CV accuracy:")
for name, pipe in [("RandomForest",rf_pipe),("HistGradientBoosting",hgb_pipe),("LogisticRegression",lr_pipe)]:
    cv = cross_val_score(pipe, X_train, y_train, cv=skf, scoring="accuracy", n_jobs=-1)
    print(f"  {name:<24} {cv.mean():.4f} ± {cv.std():.4f}")

rf_pipe.fit(X_train, y_train)
hgb_pipe.fit(X_train, y_train)
lr_pipe.fit(X_train, y_train)
print("Base models fitted.")

5-fold CV accuracy:
  RandomForest             1.0000 ± 0.0000
  HistGradientBoosting     0.9987 ± 0.0026
  LogisticRegression       1.0000 ± 0.0000
Base models fitted.


---
## Cell 10 — Stacking Ensemble + Calibration

In [10]:
meta_lr = LogisticRegression(C=1.0, max_iter=2000, multi_class="multinomial",
                              solver="lbfgs", random_state=SEED)

stacking_clf = StackingClassifier(
    estimators=[("rf",rf_pipe),("hgb",hgb_pipe),("lr",lr_pipe)],
    final_estimator=meta_lr, cv=5, stack_method="predict_proba",
    passthrough=False, n_jobs=-1)

model = CalibratedClassifierCV(stacking_clf, method="isotonic", cv="prefit")

print("Training stacking ensemble ...")
stacking_clf.fit(X_train, y_train)
model.fit(X_val, y_val)
print("Done.")
print(f"Val accuracy: {(model.predict(X_val)==y_val).mean():.4f}")

Training stacking ensemble ...
Done.
Val accuracy: 1.0000


---
## Cell 11 — Score Conversion & Inference Pipeline (`classify_new_lead`)

In [11]:
def proba_to_score(proba):
    pred_cls = int(np.argmax(proba))
    tier     = LABEL_INV[pred_cls]
    conf     = float(proba[pred_cls])
    lo, hi   = SCORE_RANGES[tier]
    return round(lo + conf*(hi-lo), 1), tier, conf

def classify_new_lead(lead, use_llm: bool = True):
    """
    End-to-end ML inference with Master Query Gating:
      - HIGH_INTENT: Ready to start, next steps, solution perfect -> HOT (Score 70-100)
      - INQUIRY    : Actively inquiring about scope / UI/UX / custom -> WARM (Score 40-69)
      - REJECTION  : Competitor chosen, unpromising, skepticism, refusal -> COLD (Score 0-39)
    """
    gate, q_sims, _, m_sims, _, llm_res = compute_query_gate(lead, check_llm=use_llm)
    raw  = extract_all_features(lead, check_llm=use_llm).reshape(1,-1)
    feat = apply_svd(raw)
    proba = model.predict_proba(feat)[0]

    llm_intent = llm_res.get("intent") if (llm_res and isinstance(llm_res, dict)) else None
    llm_reason = llm_res.get("reason", "") if llm_res else ""
    llm_source = llm_res.get("source", "Intent Engine") if llm_res else "Intent Engine"

    # ── Economic Sanity & Prank Detection (log(price)/dur ratio) ────────────
    bl = parse_budget_inr(lead.get("estimated_budget", ""))
    price_inr = bl * 100_000
    dur = parse_duration_months(lead.get("project_duration", ""))
    log_price_dur = (np.log(price_inr) / max(dur, 1.0)) if price_inr > 0 else 0.0

    svcs = lead.get("service_required", [])
    if isinstance(svcs, str): svcs = [s.strip() for s in svcs.split("|") if s.strip()]
    n_svc = len(svcs)
    min_bl = MIN_BUDGET_PER_NSVC.get(min(n_svc, 3), _MIN_4PLUS) if n_svc < 4 else _MIN_4PLUS

    # Prank condition: Absurdly low micro-budget (e.g. < 50,000 INR) for multiple services
    # While submitting high-intent queries -> Impossible / Mocking offer
    prank = bool((min_bl > 0 and bl < min_bl and (bl < 0.50 or price_inr < 50000) and n_svc >= 2) or (price_inr < 25000 and n_svc >= 2))

    is_rejection = False
    is_inquiry = False

    if prank:
        # PRANK / MOCKING OVERRIDE: Micro-budget fails economic sanity -> Force to COLD!
        p_cold = 0.95
        p_warm = 0.05
        p_hot  = 0.00
        proba  = np.array([p_cold, p_warm, p_hot])
    elif llm_intent in ["REJECTION", "COLD_EXPLORATORY"]:
        is_rejection = True
    elif llm_intent in ["INTERESTED", "HIGH_INTENT"]:
        gate = max(gate, 0.75)
        p_hot = max(proba[2], 0.70)
        p_warm = proba[1] * 0.5
        p_cold = min(proba[0], 0.05)
        proba = np.array([p_cold, p_warm, p_hot])
        proba = proba / proba.sum()
    elif llm_intent == "INQUIRY":
        is_inquiry = True
        gate = 0.55
        p_warm = max(proba[1], 0.65)
        p_hot  = min(proba[2], 0.25)
        p_cold = min(proba[0], 0.10)
        proba  = np.array([p_cold, p_warm, p_hot])
        proba  = proba / proba.sum()
    else:
        is_rejection = (q_sims[5] >= 0.28 and q_sims[5] > q_sims[0]) or q_sims[4] <= -0.5 or gate < 0.35

    if is_rejection and not prank:
        rejection_boost = 0.95 if (llm_intent in ["NOT_INTERESTED", "REJECTION", "COLD_EXPLORATORY"]) else 0.85
        p_cold = max(proba[0], 1.0 - gate, rejection_boost)
        p_warm = proba[1] * gate * 0.2
        p_hot  = proba[2] * (gate ** 2) * 0.02
        proba  = np.array([p_cold, p_warm, p_hot])
        proba  = proba / proba.sum()

    score, tier, conf = proba_to_score(proba)

    rationale = []
    if prank:
        rationale.append(f"[PRANK / MOCKING DETECTED] Micro-budget (₹{price_inr:,.0f} for {n_svc} services) fails economic feasibility (min req: ₹{min_bl*100000:,.0f}). Ratio log(P)/dur={log_price_dur:.2f}. Overridden to COLD.")
    elif llm_intent:
        rationale.append(f"[{llm_source}] Intent: {llm_intent} ({llm_reason})")
    
    if is_rejection and not prank:
        rationale.append("[REJECTION / SKEPTICISM ALERT] Client expressed negative perception, competitor selection, or refusal in query (Master Gate -> COLD)")
    elif is_inquiry and not prank:
        rationale.append("[INQUIRY / SCOPE EVALUATION] Client is actively inquiring about services & scope (Classified as WARM)")
    elif not prank:
        rationale.append(f"[Budget OK] Log(P)/dur: {log_price_dur:.2f}")

    rationale.extend([
        f"Intent Gate     : {gate:.3f}  {'[HOT]' if gate>=0.6 else '[COLD]' if is_rejection else '[WARM / INQUIRY]'}",
        f"Query sims      : hi={q_sims[0]:.3f}  lo={q_sims[1]:.3f}  NI={q_sims[5]:.3f}  kw={q_sims[4]:.3f}",
        f"Msg sims        : hi={m_sims[0]:.3f}  lo={m_sims[1]:.3f}",
        f"P(Cold)={proba[0]*100:5.1f}%  P(Warm)={proba[1]*100:5.1f}%  P(Hot)={proba[2]*100:5.1f}%",
    ])
    return {"lead":lead,"score":score,"tier":tier,"label":f"[{tier.upper()}]",
            "confidence":conf,"proba":proba,"rationale":rationale,"prank_detected":prank,
            "llm_verdict": llm_res}

def print_report(result):
    lead, W = result["lead"], 64
    def rule(c="-"): print(c*W)
    def ln(t=""): print(str(t))
    rule("="); ln("  LEAD CLASSIFICATION REPORT  [ML Ensemble v3.6 — Master Query Gating]")
    rule("=")
    ln(f"  Lead ID  : {lead.get('lead_id','N/A')}")
    ln(f"  Name     : {lead.get('name','N/A')}")
    ln(f"  Budget   : {lead.get('estimated_budget','N/A')}")
    ln(f"  Query    : \"{lead.get('query','')}\"")
    rule()
    ln(f"  SCORE    : {result['score']:5.1f} / 100  {bar(result['score'],100,32)}")
    ln(f"  TIER     : {result['label']}")
    ln(f"  CONF     : {result['confidence']*100:.1f}%")
    rule()
    for r in result["rationale"]: ln(f"  {r}")
    rule()
    if result.get("prank_detected"): ln("  ACTION: Reject / mark as prank lead. Micro-budget fails economic sanity.")
    elif result["tier"]=="Hot":      ln("  ACTION: Call within 24 hours. Assign senior rep.")
    elif result["tier"]=="Warm":     ln("  ACTION: Follow up 2-3 days. Clarify scope & custom deliverables.")
    else:                            ln("  ACTION: Add to nurture / do NOT prioritize.")
    rule("=")

print("Inference pipeline initialized with Master Query Gating.")


Inference pipeline initialized with Master Query Gating.


---
## Cell 12 — Test Set Evaluation

In [12]:
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)
accuracy = (y_pred == y_test).mean()
roc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")

print(f"Accuracy  : {accuracy:.4f}")
print(f"ROC-AUC   : {roc:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["Cold","Warm","Hot"], zero_division=0))

Accuracy  : 1.0000
ROC-AUC   : 1.0000

              precision    recall  f1-score   support

        Cold       1.00      1.00      1.00        49
        Warm       1.00      1.00      1.00        58
         Hot       1.00      1.00      1.00        58

    accuracy                           1.00       165
   macro avg       1.00      1.00      1.00       165
weighted avg       1.00      1.00      1.00       165



---
## Cell 13 — Robustness & Linguistic Contradiction Tests (Qwen3 Verified)

In [13]:
print("="*74)
print("  TESTING COMPLEX LINGUISTIC INTENT & SCOPE EXCLUSIONS")
print("="*74)

test_cases = [
    ("Ready to get started (User image 1)",
     {"lead_id":"TEST-HOT-01","urgency":"High","decision_maker":True,
      "company_size":"51-200","lead_source":"Website Contact Form",
      "estimated_budget":"Rs.10,00,000 - Rs.14,00,000",
      "service_required":["Mobile App Development","Web Development","Backend Development","Cloud Infrastructure"],
      "query":"What is the pricing for your premium plan? We are ready to get started.",
      "message":"Investors have approved the budget. We want to onboard immediately.",
      "project_description":"Society management platform.","expected_start_date":"Within 1 month"}),
    ("Solution perfect, next steps (User image 2)",
     {"lead_id":"TEST-HOT-02","urgency":"High","decision_maker":True,
      "company_size":"51-200","lead_source":"Website Contact Form",
      "estimated_budget":"Rs.10,00,000 - Rs.14,00,000",
      "service_required":["Mobile App Development","Web Development","Backend Development","Cloud Infrastructure"],
      "query":"Your solution looks perfect for our requirements. What are the next steps?",
      "message":"Investors have approved the budget. We want to onboard immediately.",
      "project_description":"Society management platform.","expected_start_date":"Within 1 month"}),
    ("Chose another provider (User image 3)",
     {"lead_id":"TEST-COLD-03","urgency":"High","decision_maker":True,
      "company_size":"51-200","lead_source":"Website Contact Form",
      "estimated_budget":"Rs.10,00,000 - Rs.14,00,000",
      "service_required":["Mobile App Development","Web Development","Backend Development","Cloud Infrastructure"],
      "query":"We have decided to go with another provider.",
      "message":"Investors have approved the budget. We want to onboard immediately.",
      "project_description":"Society management platform.","expected_start_date":"Within 1 month"}),
    ("Services not promising",
     {"lead_id":"TEST-COLD-NP","urgency":"High","decision_maker":True,
      "company_size":"51-200","lead_source":"Website Contact Form",
      "estimated_budget":"Rs.10,00,000 - Rs.14,00,000",
      "service_required":["Mobile App Development","Web Development","Backend Development","Cloud Infrastructure"],
      "query":"Let us decide on the scope. But to be specific, we don't think your services to be promising.",
      "message":"Investors have approved the budget. We want to onboard immediately.",
      "project_description":"Society management platform.","expected_start_date":"Within 1 month"}),
    ("Scope & extra services inquiry (UI/UX design)",
     {"lead_id":"TEST-WARM-01","urgency":"High","decision_maker":True,
      "company_size":"51-200","lead_source":"Website Contact Form",
      "estimated_budget":"Rs.10,00,000 - Rs.14,00,000",
      "service_required":["Mobile App Development","Web Development","Backend Development","Cloud Infrastructure"],
      "query":"we don't want website codes and files. Can you provide extra services like UI/UX design?",
      "message":"Investors have approved the budget. We want to onboard immediately.",
      "project_description":"Society management platform.","expected_start_date":"Within 1 month"}),
    ("Contrastive scope (WANT full deployment)",
     {"lead_id":"TEST-HOT-03","urgency":"High","decision_maker":True,
      "company_size":"51-200","lead_source":"Website Contact Form",
      "estimated_budget":"Rs.10,00,000 - Rs.14,00,000",
      "service_required":["Mobile App Development","Web Development","Backend Development","Cloud Infrastructure"],
      "query":"we don't want website codes and files. We want full deployment.",
      "message":"Investors have approved the budget. We want to onboard immediately.",
      "project_description":"Society management platform.","expected_start_date":"Within 1 month"}),
    ("Micro-budget prank lead (Rs. 5K - 10K for 4 services)",
     {"lead_id":"TEST-PRANK-01","urgency":"High","decision_maker":True,
      "company_size":"51-200","lead_source":"Website Contact Form",
      "estimated_budget":"₹5,000 - ₹10,000",
      "service_required":["Mobile App Development","Web Development","Backend Development","Cloud Infrastructure"],
      "query":"We have finalized our budget and would like to move forward.",
      "message":"Investors have approved the budget. We want to onboard immediately. Have detailed BRD ready.",
      "project_description":"Society management: visitor gate, Razorpay maintenance, complaint ticketing, analytics dashboard.","expected_start_date":"Within 1 month"}),
]

for desc, ld in test_cases:
    res = classify_new_lead(ld, use_llm=True)
    print(f"  Scenario: {desc}")
    print(f"  Query   : '{ld['query']}'")
    print(f"  Result  : {res['label']} (Score: {res['score']:5.1f} | Conf: {res['confidence']*100:.1f}%)")
    llm_v = res.get('llm_verdict', {})
    if llm_v:
        print(f"  Intent  : [{llm_v.get('source','Engine')}] {llm_v.get('intent')} | Reason: {llm_v.get('reason')}")
    print(f"  Probas  : P(Cold)={res['proba'][0]*100:5.1f}%  P(Warm)={res['proba'][1]*100:5.1f}%  P(Hot)={res['proba'][2]*100:5.1f}%")
    print("-"*74)


  TESTING COMPLEX LINGUISTIC INTENT & SCOPE EXCLUSIONS
  Scenario: Ready to get started (User image 1)
  Query   : 'What is the pricing for your premium plan? We are ready to get started.'
  Result  : [HOT] (Score:  99.7 | Conf: 99.1%)
  Intent  : [Intent Engine] HIGH_INTENT | Reason: Client indicates readiness to start or asks next steps (matched: '\bready\s+to\s+(get\s+started|start|proceed|move\s+forward|onboard|buy|hire)\b')
  Probas  : P(Cold)=  0.0%  P(Warm)=  0.9%  P(Hot)= 99.1%
--------------------------------------------------------------------------
  Scenario: Solution perfect, next steps (User image 2)
  Query   : 'Your solution looks perfect for our requirements. What are the next steps?'
  Result  : [HOT] (Score: 100.0 | Conf: 100.0%)
  Intent  : [Intent Engine] HIGH_INTENT | Reason: Client indicates readiness to start or asks next steps (matched: '\bwhat\s+are\s+the\s+next\s+steps\b')
  Probas  : P(Cold)=  0.0%  P(Warm)=  0.0%  P(Hot)=100.0%
-----------------------------

---
## Cell 14 — Feature Importance

In [14]:
rf_clf      = rf_pipe.named_steps["clf"]
importances = rf_clf.feature_importances_
feat_names  = FEAT_NAMES_NOSVD + [f"svd_{i}" for i in range(SVD_COMPONENTS)]

top_n   = 20
top_idx = np.argsort(importances)[::-1][:top_n]
QUERY_FEAT_IDX = set(range(17, 23))

print(f"Top {top_n} Features by RF Importance:")
print("-"*60)
for rank, idx in enumerate(top_idx, 1):
    nm  = feat_names[idx] if idx < len(feat_names) else f"feat_{idx}"
    imp = importances[idx]
    tag = "<-- QUERY SIM" if idx in QUERY_FEAT_IDX else ""
    print(f"  {rank:<3} {nm:<24} {imp:>8.5f}  {bar(imp,importances[top_idx[0]],20)} {tag}")


Top 20 Features by RF Importance:
------------------------------------------------------------
  1   budget_norm               0.11709  [####################] 
  2   gate_x_budget             0.09864  [#################...] 
  3   budget_log                0.09222  [################....] 
  4   num_svc                   0.08086  [##############......] 
  5   budget_per_svc            0.07142  [############........] 
  6   svd_1                     0.05844  [##########..........] 
  7   start_urg                 0.05297  [#########...........] 
  8   gate_x_urgency            0.05225  [#########...........] 
  9   price_per_month           0.04890  [########............] 
  10  intent_gate               0.03930  [#######.............] 
  11  urg_high                  0.02890  [#####...............] 
  12  size_log                  0.02844  [#####...............] 
  13  gate_x_dm                 0.02835  [#####...............] 
  14  svd_2                     0.02405  [####..............

---
## Cell 15 — Calibration (Brier Score)

In [15]:
for c in range(3):
    bs = brier_score_loss((y_test==c).astype(int), y_proba[:,c])
    print(f"  Brier [{LABEL_INV[c]}]  : {bs:.4f}  {bar(1-bs,1.0,20)}")

  Brier [Cold]  : 0.0000  [####################]
  Brier [Warm]  : 0.0001  [####################]
  Brier [Hot]  : 0.0001  [####################]


---
## Cell 16 — Leaderboard + Validation on data.json

In [16]:
def batch_score(leads_list):
    return sorted([classify_new_lead(ld, use_llm=False) for ld in leads_list],
                  key=lambda r: r["score"], reverse=True)

def print_leaderboard(results, max_rows=25, title="LEADERBOARD"):
    W = 82
    print("="*W)
    print(f"  {title}")
    print("="*W)
    print(f"  {'#':<3} {'ID':<14} {'Tier':<8} {'Score':>6} {'Conf%':>6}  {'Name':<20} True  Status")
    print("-"*W)
    correct = total = 0
    for i,r in enumerate(results[:max_rows],1):
        ld = r["lead"]; true_lb = ld.get("label","?")
        hit = (r["tier"]==true_lb)
        if true_lb in LABEL_MAP: total+=1; correct+=int(hit)
        ok = "OK" if hit else ("MISS" if true_lb in LABEL_MAP else "")
        flag = "[!]" if r.get("prank_detected") else "   "
        print(f"  {i:<3} {ld.get('lead_id',''):<14} {r['label']:<8} "
              f"{r['score']:>6.1f} {r['confidence']*100:>5.1f}%  "
              f"{str(ld.get('name',''))[:18]:<20} [{true_lb}] {ok} {flag}")
    if len(results)>max_rows: print(f"  ... {len(results)-max_rows} more ...")
    print("="*W)
    if total: print(f"  Accuracy: {correct}/{total} ({correct/total*100:.0f}%)")

import random as rnd; rnd.seed(42)
print_leaderboard(batch_score(rnd.sample(labeled_leads,50)), title="SAMPLE (50 leads)")

if Path(VALIDATION_FILE).exists():
    print()
    val20 = [ld for ld in load_dataset(VALIDATION_FILE) if ld.get("label") in LABEL_MAP]
    print_leaderboard(batch_score(val20), title="VALIDATION: data.json (20 leads)")

  SAMPLE (50 leads)
  #   ID             Tier      Score  Conf%  Name                 True  Status
----------------------------------------------------------------------------------
  1   LD-GEN-0564    [HOT]     100.0 100.0%  Preeti Patel         [Hot] OK    
  2   LD-GEN-0286    [HOT]     100.0 100.0%  Shweta Sharma        [Hot] OK    
  3   LD-GEN-0865    [HOT]     100.0 100.0%  Shweta Malhotra      [Hot] OK    
  4   LD-GEN-0066    [HOT]     100.0 100.0%  Kajal Sen            [Hot] OK    
  5   LD-GEN-0860    [HOT]     100.0 100.0%  Jyoti Patel          [Hot] OK    
  6   LD-GEN-0452    [HOT]     100.0 100.0%  Seema Thakur         [Hot] OK    
  7   LD-GEN-0014    [HOT]     100.0 100.0%  Kajal Menon          [Hot] OK    
  8   LD-GEN-0327    [HOT]     100.0 100.0%  Naresh Kumar         [Hot] OK    
  9   LD-GEN-0319    [HOT]     100.0 100.0%  Ravi Thakur          [Hot] OK    
  10  LD-GEN-0690    [HOT]     100.0 100.0%  Nisha Singh          [Hot] OK    
  11  LD-GEN-0190    [HOT]  

---
## Cell 17 — New Lead Inference Demo (User's Exact Query Handled by Qwen3)

In [21]:
# ── NEW LEAD (You can edit 'query' and any other attributes directly here) ────
new_lead = {
    "lead_id":             "LD-PROD-001",
    "name":                "Amit Verma",
    "company_name":        "SkyRocket Ventures Pvt. Ltd.",
    "company_size":        "51-200",
    "industry":            "Real Estate",
    "designation":         "CEO",
    # ── QUERY (Edit your test query directly here) ────────────────────────
    "query":               "We might be interested, but our budget has not been finalized yet.",
    "message":             "Investors have approved the budget. We want to onboard immediately. Have detailed BRD ready.",
    "project_description": "Society management: visitor gate, Razorpay maintenance, complaint ticketing, analytics dashboard.",
    "service_required":    ["Mobile App Development","Web Development","Backend Development","Cloud Infrastructure"],
    "estimated_budget":    "₹5,00,000 - ₹10,00,000",
    "expected_start_date": "Within 1 month",
    "urgency":             "High",
    "decision_maker":      True,
    "lead_source":         "Website Contact Form",
}

# Classify using Master Query Gating + ML Stacking Ensemble
result = classify_new_lead(new_lead, use_llm=True)
print_report(result)


  LEAD CLASSIFICATION REPORT  [ML Ensemble v3.6 — Master Query Gating]
  Lead ID  : LD-PROD-001
  Name     : Amit Verma
  Budget   : ₹5,00,000 - ₹10,00,000
  Query    : "We might be interested, but our budget has not been finalized yet."
----------------------------------------------------------------
  SCORE    :  90.5 / 100  [#############################...]
  TIER     : [HOT]
  CONF     : 68.4%
----------------------------------------------------------------
  [Semantic Engine] Intent: HIGH_INTENT (Semantic match to commercial demand (0.66))
  [Budget OK] Log(P)/dur: 4.61
  Intent Gate     : 0.750  [HOT]
  Query sims      : hi=0.450  lo=0.406  NI=0.180  kw=0.000
  Msg sims        : hi=0.421  lo=0.282
  P(Cold)=  4.8%  P(Warm)= 26.8%  P(Hot)= 68.4%
----------------------------------------------------------------
  ACTION: Call within 24 hours. Assign senior rep.
